engagement exists

engagement is a function that we dont know and we estimate

-4 a 4

avg(engagement) = 0

each behavioral metric score is multiplied by an engagement strength, how much that behavior contributes to engagement.

negative weight means disengagement

engagement is the sum of the contribution of each behavior

engagement is a dot product of metrics and weights

engagement levels are transformed by the s curve to give retention and churn probabilities

esos productos punto van de -4 a 4

sigmoid(engagement)=prob retention

the weights are called coefficients

Engagement weights and the S curve offset

if you have sigma on one metric then the probability change retention probability if you move one to the right on that metric.


In [10]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LogisticRegression
from math import exp
import pickle


In [33]:
grouped_data = pd.read_csv('groupscore.csv',index_col=[0,1])
grouped_data

,,metric_group_0,metric_group_1,metric_group_2,dislike_per_month,account_tenure,reply_per_message,dislike_pcnt,newfriend_pcnt_chng,days_since_newfriend,is_churn
account_id,observation_date,,,,,,,,,,
27,2020-03-01,0.583781,0.835844,-2.541958,-1.074832,-0.600623,-1.524524,-0.789858,3.220524,-0.073812,False
51,2020-03-01,0.145983,1.701262,0.883564,-0.593460,-0.600623,1.034653,-0.468472,-0.436188,-0.683207,False
95,2020-03-01,0.226257,-0.875414,-0.509761,0.758881,-0.600623,0.506809,-0.085544,0.232470,-0.683207,False
123,2020-03-01,-0.491911,-0.875414,-1.218812,0.960141,-0.600623,-1.088607,1.162712,1.257860,-0.326734,False
189,2020-03-01,-1.055549,-0.875414,-2.911179,-1.074832,-0.600623,-1.524524,0.817071,-0.167825,1.144979,True
...,...,...,...,...,...,...,...,...,...,...,...
13132,2020-05-10,-1.251514,-0.875414,-0.101065,-0.803737,-1.722229,-0.519187,-0.453864,-0.167825,-1.292602,False
13141,2020-05-10,-1.143124,-0.875414,-0.552287,-1.456919,-1.722229,0.105805,0.093953,-0.167825,1.296065,False
13155,2020-05-10,-1.796094,-0.875414,0.401347,-1.456919,-1.722229,0.922067,0.495685,-0.167825,-1.292602,False


In [32]:
y = grouped_data['is_churn'].astype(np.bool)
#is retention 
y = ~y
y

account_id  observation_date
27          2020-03-01           True
51          2020-03-01           True
95          2020-03-01           True
123         2020-03-01           True
189         2020-03-01          False
                                ...  
13132       2020-05-10           True
13141       2020-05-10           True
13155       2020-05-10           True
13171       2020-05-10           True
13248       2020-05-10           True
Name: is_churn, Length: 25806, dtype: bool

In [35]:
X = grouped_data.drop(['is_churn'], axis=1)
X

,,metric_group_0,metric_group_1,metric_group_2,dislike_per_month,account_tenure,reply_per_message,dislike_pcnt,newfriend_pcnt_chng,days_since_newfriend
account_id,observation_date,,,,,,,,,
27,2020-03-01,0.583781,0.835844,-2.541958,-1.074832,-0.600623,-1.524524,-0.789858,3.220524,-0.073812
51,2020-03-01,0.145983,1.701262,0.883564,-0.593460,-0.600623,1.034653,-0.468472,-0.436188,-0.683207
95,2020-03-01,0.226257,-0.875414,-0.509761,0.758881,-0.600623,0.506809,-0.085544,0.232470,-0.683207
123,2020-03-01,-0.491911,-0.875414,-1.218812,0.960141,-0.600623,-1.088607,1.162712,1.257860,-0.326734
189,2020-03-01,-1.055549,-0.875414,-2.911179,-1.074832,-0.600623,-1.524524,0.817071,-0.167825,1.144979
...,...,...,...,...,...,...,...,...,...,...
13132,2020-05-10,-1.251514,-0.875414,-0.101065,-0.803737,-1.722229,-0.519187,-0.453864,-0.167825,-1.292602
13141,2020-05-10,-1.143124,-0.875414,-0.552287,-1.456919,-1.722229,0.105805,0.093953,-0.167825,1.296065
13155,2020-05-10,-1.796094,-0.875414,0.401347,-1.456919,-1.722229,0.922067,0.495685,-0.167825,-1.292602


In [47]:
#lasso regression l1
#automatic feature selection

retain_reg = LogisticRegression(fit_intercept=True, solver='liblinear', penalty='l1')
retain_reg.fit(X, y)

/home/alejandro-armenta/Documents/churn/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/alejandro-armenta/Documents/churn/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [77]:
def s_curve(x):
    return 1.0 / (1.0 + np.exp(-x))
    

#bias term predicted
average_retain = s_curve(retain_reg.intercept_)
average_retain

one_stdev_retains = np.array([
    s_curve(retain_reg.intercept_ + c) for c in retain_reg.coef_[0]
])

one_stdev_impact = one_stdev_retains - average_retain

average_retain
one_stdev_impact


array([[ 0.00180537],
       [-0.00200771],
       [ 0.00572376],
       [ 0.00225756],
       [ 0.00478077],
       [-0.00074032],
       [-0.0041946 ],
       [ 0.00106333],
       [ 0.00191243]])

In [92]:
group_lists = pd.read_csv('groupmets.csv', index_col=0)
group_lists

,metrics
metric_group_0,adview_per_month|adview_per_post|like_per_mont...
metric_group_1,unfriend_28day_avg_84day_obs|unfriend_28day_av...
metric_group_2,message_per_month|reply_per_month
dislike_per_month,dislike_per_month
account_tenure,account_tenure
reply_per_message,reply_per_message
dislike_pcnt,dislike_pcnt
newfriend_pcnt_chng,newfriend_pcnt_chng
days_since_newfriend,days_since_newfriend


In [100]:
coef_df = pd.DataFrame.from_dict(
    {
        'group_metric_offset':np.append(group_lists.index, 'offset'),
        'weight': np.append(retain_reg.coef_[0],retain_reg.intercept_),
        'retain_impact': np.append(one_stdev_impact, average_retain),
        'group_metrics': np.append(group_lists['metrics'],'baseline')
    }
)
coef_df

,group_metric_offset,weight,retain_impact,group_metrics
0,metric_group_0,0.148768,0.001805,adview_per_month|adview_per_post|like_per_mont...
1,metric_group_1,-0.143511,-0.002008,unfriend_28day_avg_84day_obs|unfriend_28day_av...
2,metric_group_2,0.573622,0.005724,message_per_month|reply_per_month
3,dislike_per_month,0.189678,0.002258,dislike_per_month
4,account_tenure,0.454051,0.004781,account_tenure
5,reply_per_message,-0.055274,-0.000740,reply_per_message
6,dislike_pcnt,-0.279999,-0.004195,dislike_pcnt
7,newfriend_pcnt_chng,0.084989,0.001063,newfriend_pcnt_chng
8,days_since_newfriend,0.158308,0.001912,days_since_newfriend
9,offset,4.313392,0.986789,baseline


In [130]:
coef_df.sort_values(by=['weight'],ascending=False, inplace=True)
coef_df.to_csv('logreg_summary.csv',index=False)

In [107]:
with open('model.pkl', 'wb') as fid:
    print(fid)
    pickle.dump(retain_reg, fid)

<_io.BufferedWriter name='model.pkl'>


In [118]:
predictions = retain_reg.predict_proba(X)
retain_reg.classes_

array([False,  True])

In [122]:
predict_df = pd.DataFrame(predictions, index=X.index, columns=['churn_prob','retain_prob'])
predict_df

,,churn_prob,retain_prob
account_id,observation_date,,
27,2020-03-01,0.051513,0.948487
51,2020-03-01,0.015654,0.984346
95,2020-03-01,0.018724,0.981276
123,2020-03-01,0.033378,0.966622
189,2020-03-01,0.103589,0.896411
...,...,...,...
13132,2020-05-10,0.039267,0.960733
13141,2020-05-10,0.045106,0.954894
13155,2020-05-10,0.050447,0.949553


In [124]:
predict_df.to_csv("predictions.csv", header=True)